In [1]:
import os
import numpy as np
import pandas as pd

RAW_PATH = "../data/raw/city_day.csv"
OUT_PATH = "../data/processed/city_day_cleaned.csv"

os.makedirs("../data/processed", exist_ok=True)

df = pd.read_csv(RAW_PATH)

print("Raw shape:", df.shape)

Raw shape: (29531, 16)


In [2]:
df = df.copy()

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

df = df.dropna(subset=["Date"])

# Remove exact duplicate rows
df = df.drop_duplicates()

# If duplicate City-Date records exist, keep the first after inspecting them.
duplicate_city_date = df.duplicated(["City", "Date"]).sum()
print("Duplicate City-Date pairs:", duplicate_city_date)

if duplicate_city_date > 0:
    df = df.drop_duplicates(["City", "Date"], keep="first")

# Convert target/pollutants to numeric
pollutants = [
    "PM2.5", "PM10", "NO", "NO2", "NOx", "NH3",
    "CO", "SO2", "O3", "Benzene", "Toluene", "Xylene"
]

for col in pollutants + ["AQI"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Physical constraint: pollutant concentrations cannot be negative.
for col in pollutants:
    df.loc[df[col] < 0, col] = np.nan

# AQI is the supervised target, so missing-target rows are removed.
df = df.dropna(subset=["AQI"])

print("Cleaned shape:", df.shape)

Duplicate City-Date pairs: 0
Cleaned shape: (24850, 16)


In [3]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["DayOfWeek"] = df["Date"].dt.dayofweek
df["DayOfYear"] = df["Date"].dt.dayofyear
df["Quarter"] = df["Date"].dt.quarter
df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Summer"
    if month in [6, 7, 8, 9]:
        return "Monsoon"
    return "PostMonsoon"

df["Season"] = df["Month"].apply(get_season)

In [6]:
df["PM_Ratio"] = df["PM2.5"] / (df["PM10"] + 1e-6)
df["NOx_NO2_Ratio"] = df["NOx"] / (df["NO2"] + 1e-6)
df["Particulate_Load"] = df["PM2.5"] + df["PM10"]

gas_cols = ["NO", "NO2", "NOx", "NH3", "SO2", "O3"]
df["Gas_Pollution"] = df[gas_cols].sum(axis=1, min_count=1)

df["Total_Pollution"] = df[pollutants].sum(axis=1, min_count=1)

print(df.shape)
display(df.head())

(24850, 29)


,City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,...,DayOfWeek,DayOfYear,Quarter,IsWeekend,Season,PM_Ratio,NOx_NO2_Ratio,Particulate_Load,Gas_Pollution,Total_Pollution
28,Ahmedabad,2015-01-29,83.13,NaN,6.93,28.71,33.72,NaN,6.93,49.52,...,3,29,1,0,Winter,NaN,1.174504,NaN,178.64,271.86
29,Ahmedabad,2015-01-30,79.84,NaN,13.85,28.68,41.08,NaN,13.85,48.49,...,4,30,1,0,Winter,NaN,1.432357,NaN,229.17,327.71
30,Ahmedabad,2015-01-31,94.52,NaN,24.39,32.66,52.61,NaN,24.39,67.39,...,5,31,1,1,Winter,NaN,1.610839,NaN,288.38,415.21
31,Ahmedabad,2015-02-01,135.99,NaN,43.48,42.08,84.57,NaN,43.48,75.23,...,6,32,1,1,Winter,NaN,2.009743,NaN,348.06,553.84
32,Ahmedabad,2015-02-02,178.33,NaN,54.56,35.31,72.80,NaN,54.56,55.04,...,0,33,1,0,Winter,NaN,2.061739,NaN,325.09,594.11


In [ ]:
engineered = [
    "PM_Ratio",
    "NOx_NO2_Ratio",
    "Particulate_Load",
    "Gas_Pollution",
    "Total_Pollution"
]

display(df[engineered].describe().T)

# Infinite values can arise from ratios.
df = df.replace([np.inf, -np.inf], np.nan)

print("Remaining missing values:")
display(df.isna().sum().sort_values(ascending=False).head(20))

,count,mean,std,min,25%,50%,75%,max
PM_Ratio,17146.0,0.694898,6.840658,0.002797,0.375705,0.472130,0.575126,362.316535
NOx_NO2_Ratio,22741.0,2.463704,60.748940,0.000000,0.684075,1.005278,1.524437,8889.111089
Particulate_Load,17146.0,175.000372,134.866130,1.390000,84.262500,140.625000,218.980000,1436.070000
Gas_Pollution,24798.0,141.410589,88.115347,0.000000,83.230000,119.570000,174.597500,904.880000
Total_Pollution,24801.0,305.961758,198.367094,0.000000,171.720000,256.900000,380.330000,2001.460000


Remaining missing values:


Xylene              15372
PM_Ratio             7704
Particulate_Load     7704
PM10                 7086
NH3                  6536
Toluene              5826
Benzene              3535
NOx_NO2_Ratio        2109
NOx                  1857
O3                    807
PM2.5                 678
SO2                   605
CO                    445
NO2                   391
NO                    387
Gas_Pollution          52
Total_Pollution        49
City                    0
Date                    0
AQI_Bucket              0
dtype: int64

In [8]:
TARGET = "AQI"

DROP_FOR_MODEL = [
    "AQI",
    "AQI_Bucket",
    "Date"
]

FEATURE_COLUMNS = [
    col for col in df.columns
    if col not in DROP_FOR_MODEL
]

print("Number of model features:", len(FEATURE_COLUMNS))
print(FEATURE_COLUMNS)

Number of model features: 26
['City', 'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene', 'Year', 'Month', 'Day', 'DayOfWeek', 'DayOfYear', 'Quarter', 'IsWeekend', 'Season', 'PM_Ratio', 'NOx_NO2_Ratio', 'Particulate_Load', 'Gas_Pollution', 'Total_Pollution']


In [9]:
train_df = df[df["Date"] < "2020-01-01"].copy()
test_df = df[df["Date"] >= "2020-01-01"].copy()

print("Train:", train_df.shape)
print("Test :", test_df.shape)

print("\nTrain period:")
print(train_df["Date"].min(), "to", train_df["Date"].max())

print("\nTest period:")
print(test_df["Date"].min(), "to", test_df["Date"].max())

Train: (20429, 29)
Test : (4421, 29)

Train period:
2015-01-01 00:00:00 to 2019-12-31 00:00:00

Test period:
2020-01-01 00:00:00 to 2020-07-01 00:00:00


In [10]:
df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
print("Final shape:", df.shape)

Saved: ../data/processed/city_day_cleaned.csv
Final shape: (24850, 29)
